# 19장 텍스트 데이터 — 글자를 숫자로

**2단계 · 주석 달기** — 『법률가를 위한 인공지능 코딩 자습서』 실습 노트북

[![Colab에서 열기](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch19_2_comment.ipynb)

이 노트북에는 **주석을 지운 코드**가 들어 있습니다. 아래 표의 「주석 달기」 과제대로 내 말로 주석을 달아 보세요. 주석을 다는 동안 코드를 한 줄씩 다시 읽게 됩니다.

- 코드 셀 첫 줄의 `%%writefile 파일이름` 은 "이 셀의 내용을 그 파일로 저장하라"는 뜻입니다. 셀을 실행하면 내가 고친 내용이 파일에 저장됩니다.
- 그 아래 확인 셀을 실행해, 주석을 단 뒤에도 코드가 전과 똑같이 도는지 봅니다.

| 단계 | 할 일 | 노트북 |
|---|---|---|
| 1. 따라 하기 | `fetch_statutes.py` 로 데이터를 받고 `p01`~`p03`, `statutes_dataset.py` 를 치고 실행 | [ch19_1_follow.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch19_1_follow.ipynb) |
| 2. 주석 달기 | `Vocab.encode` 와 `BagOfWords.forward` 각 줄 뒤에 shape을 주석으로 | [ch19_2_comment.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch19_2_comment.ipynb) **(지금 이 노트북)** |
| 3. 다시 만들기 | 주석만 남긴 파일에서 `StatuteDataset` 복원. 그다음 14장 패키지의 `data/` 에 `statutes.py` 로 넣고 `--dataset statutes --model bow` 로 학습되게 등록 | [ch19_3_build.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch19_3_build.ipynb) |

In [ ]:
# 준비 셀 — 맨 먼저 한 번 실행하세요. 실습 코드와 데이터를 받아 옵니다.
import os, sys
if "google.colab" in sys.modules:                       # Colab 에서 열었을 때
    ROOT = "/content/AI-Coding-for-Lawyers"
    if not os.path.isdir(ROOT):
        !git clone -q --depth 1 https://github.com/BowonKwon/AI-Coding-for-Lawyers.git {ROOT}
else:                                                   # 내 컴퓨터에서 열었을 때 (저장소 폴더 안)
    ROOT = os.getcwd()
    while not os.path.isdir(os.path.join(ROOT, "notebooks")) and os.path.dirname(ROOT) != ROOT:
        ROOT = os.path.dirname(ROOT)
os.environ.setdefault("JUCOS_DATA", os.path.join(ROOT, "data"))   # 데이터를 두는 폴더
os.chdir(os.path.join(ROOT, "code", "19"))                      # 이 장의 코드 폴더로 이동
print("준비 끝. 작업 폴더: code/19")

## `statutes_dataset.py` — 대상: `Vocab.encode`

In [ ]:
%%writefile statutes_dataset.py
# 19.4 텍스트 Dataset: 조문 → (토큰 번호 텐서, 법령 번호). 15장의 ImageFolderDataset 과 같은 뼈대.
import os
from collections import Counter
import torch
from torch.utils.data import Dataset, DataLoader

DATA = os.path.join(os.environ.get("JUCOS_DATA", "./data"), "statutes")
PAD, UNK = "<pad>", "<unk>"


def tokenize(s):
    return s.split()


class Vocab:
    def __init__(self, sentences, min_count=2, max_size=8000):
        counter = Counter(t for s in sentences for t in tokenize(s))
        self.itos = [PAD, UNK] + [t for t, c in counter.most_common(max_size) if c >= min_count]
        self.stoi = {t: i for i, t in enumerate(self.itos)}

    def __len__(self):
        return len(self.itos)

    def encode(self, s, max_len):
        ids = [self.stoi.get(t, 1) for t in tokenize(s)][:max_len]
        return ids + [0] * (max_len - len(ids))


class StatuteDataset(Dataset):
    """여러 법령의 조문을 모아 '어느 법령의 조문인가' 를 맞히는 분류 데이터셋."""

    def __init__(self, laws, split="train", vocab=None, max_len=40, test_ratio=0.2, seed=0):
        self.classes = list(laws)
        items = []
        for label, name in enumerate(laws):
            with open(os.path.join(DATA, f"{name}.txt"), encoding="utf-8") as f:
                lines = [l.strip() for l in f if l.strip()]
            lines = [l for l in lines if len(tokenize(l)) >= 4]     # '제786조 삭제' 같은 빈 조문은 뺀다
            items += [(l, label) for l in lines]
        g = torch.Generator().manual_seed(seed)             # 같은 씨앗이면 같은 분할
        perm = torch.randperm(len(items), generator=g).tolist()
        n_test = int(len(items) * test_ratio)
        idx = perm[n_test:] if split == "train" else perm[:n_test]
        self.items = [items[i] for i in idx]
        self.vocab = vocab or Vocab([s for s, _ in self.items])   # 어휘는 학습 데이터로만 만든다
        self.max_len = max_len

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        text, label = self.items[i]
        return torch.tensor(self.vocab.encode(text, self.max_len)), label


if __name__ == "__main__":
    laws = ["민법", "형법", "민사소송법", "형사소송법"]
    train = StatuteDataset(laws, "train")
    test = StatuteDataset(laws, "test", vocab=train.vocab)        # 시험 데이터는 학습 어휘를 그대로 쓴다
    print("학습", len(train), "시험", len(test), "| 어휘", len(train.vocab), "| 클래스", train.classes)
    x, y = train[0]
    print("train[0] ->", tuple(x.shape), x.dtype, "| label", y, train.classes[y])
    print("  원문:", train.items[0][0][:50])
    print("  복원:", " ".join(train.vocab.itos[i] for i in x.tolist() if i != 0)[:50])
    xb, yb = next(iter(DataLoader(train, batch_size=8, shuffle=True)))
    print("배치:", tuple(xb.shape), tuple(yb.shape))
    counts = Counter(train.classes[l] for _, l in train.items)
    print("학습 클래스별:", dict(counts))

## `p03_train_bow.py` — 대상: `BagOfWords.forward`

In [ ]:
%%writefile p03_train_bow.py
# 19.5 첫 텍스트 분류기: 임베딩 → 평균 → 선형층 ("단어 주머니" 모델). 12장 루프 그대로.
import time
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from statutes_dataset import StatuteDataset

LAWS = ["민법", "형법", "민사소송법", "형사소송법"]
torch.manual_seed(0)
train_set = StatuteDataset(LAWS, "train")
test_set = StatuteDataset(LAWS, "test", vocab=train_set.vocab)
train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
test_loader = DataLoader(test_set, batch_size=256)


class BagOfWords(nn.Module):
    def __init__(self, vocab_size, dim, num_classes):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, dim, padding_idx=0)   # <pad> 의 벡터는 0 으로 고정
        self.fc = nn.Linear(dim, num_classes)

    def forward(self, ids):
        vec = self.emb(ids)
        mask = (ids != 0).unsqueeze(-1).float()
        sent = (vec * mask).sum(1) / mask.sum(1).clamp(min=1)
        return self.fc(sent)


model = BagOfWords(len(train_set.vocab), dim=64, num_classes=len(LAWS))
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


def accuracy():
    model.eval()
    c = 0
    with torch.no_grad():
        for x, y in test_loader:
            c += (model(x).argmax(1) == y).sum().item()
    model.train()
    return c / len(test_set)


print(f"학습 {len(train_set)} / 시험 {len(test_set)} / 어휘 {len(train_set.vocab)} / 파라미터 {sum(p.numel() for p in model.parameters()):,}")
print(f"학습 전 정확도: {accuracy():.3f}")
t0 = time.time()
for epoch in range(1, 11):
    for x, y in train_loader:
        optimizer.zero_grad()
        loss = loss_fn(model(x), y)
        loss.backward()
        optimizer.step()
    if epoch % 2 == 0:
        print(f"에포크 {epoch:2d}: 손실 {loss.item():.4f}  시험 정확도 {accuracy():.3f}")
print(f"경과 {time.time() - t0:.0f}초")

# 어느 조문을 어느 법으로 보는가: 시험 데이터 몇 개
model.eval()
with torch.no_grad():
    for i in range(5):
        x, y = test_set[i]
        pred = model(x.unsqueeze(0)).argmax(1).item()
        mark = "" if pred == y else "  ← 틀림"
        print(f"  [{LAWS[y]} → 예측 {LAWS[pred]}]{mark} {test_set.items[i][0][:45]}…")

### 확인

In [ ]:
!python p03_train_bow.py

### 원래대로 되돌리기

파일을 책의 원래 코드로 되돌리고 싶으면 아래 셀을 실행하세요. (Colab 에서 받은 저장소 기준입니다.)

In [ ]:
!git checkout -- statutes_dataset.py p03_train_bow.py